# Initial Data Treatment

In [1]:
import os
import re
import unicodedata
import pandas as pd
import polars as pl
from lingua import LanguageDetectorBuilder, Language
from nltk.corpus import stopwords

# ingestion path
CHANNELS_PATH = '../CrawlerYTGabriel/files/channels_info.csv'
VIDEOS_PATH = '../CrawlerYTGabriel/files/videos_info.csv'
COMMENTS_PATH = '../CrawlerYTGabriel/files/comments_info.csv'
KEY_TERMS_PATH = '../data/cleaned/key_terms.txt'
EXCLUDE_TERMS_PATH = '../data/cleaned/exclude_terms.txt'
STRONG_EXCLUDE_TERMS_PATH = '../data/cleaned/strong_exclude_terms.txt'

# saving path
CHANNELS_CLEANED_PATH = '../data/cleaned/cleaned_channels_info.csv'
VIDEOS_CLEANED_PATH = '../data/cleaned/cleaned_videos_info.csv'
COMMENTS_CLEANED_PATH = '../data/cleaned/cleaned_comments_info.csv'

# creates cleaned folder in case it doesn't exists
os.makedirs('../data/cleaned', exist_ok=True)

# relevant channels info
COLS_CHANNELS = ['channel_id', 'title', 'description', 'published_at', 'country', 'view_count', 'comment_count', 'subscriber_count', 'video_count', 'keywords', 'profile_picture_url']

# relevant videos info
COLS_VIDEOS = ['video_id', 'title', 'description', 'channel_id', 'published_at', 'category_id', 'tags', 'view_count', 'like_count', 'comment_count', 'duration', 'licensed_content', 'is_made_for_kids', 'thumbnail_url', 'default_audio_language', 'default_language', 'topicCategories', 'is_short']

# relevant comments info
COLS_COMMENTS = ['video_id', 'comment_id', 'author_profile_image_url', 'author_channel_url', 'author_channel_id', 'comment', 'published_at', 'like_count', 'is_reply', 'parent_id', 'channel_id']

# model for language detection
detector = LanguageDetectorBuilder.from_all_languages().build()

stopwords_pt = list(set(stopwords.words('portuguese')))

In [2]:
def padronize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    normalized_text = unicodedata.normalize('NFKD', text)
    text_without_accents = "".join([c for c in normalized_text if not unicodedata.combining(c)])
    clean_text = re.sub(r'[^\w\s]', ' ', text_without_accents)
    clean_text = re.sub(r'\s+', ' ', clean_text)
    
    return clean_text.strip()

def read_terms_from_file(file_path: str) -> list:
    terms = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line:
                continue
                
            parts = [p.strip() for p in line.split(",") if p.strip()]
            
            for term in parts:
                term_limpo = padronize_text(term.lower())
                terms.append(term_limpo)
    
    return terms

KEY_TERMS = read_terms_from_file(KEY_TERMS_PATH)
EXCLUDE_TERMS = read_terms_from_file(EXCLUDE_TERMS_PATH)
STRONG_EXCLUDE_TERMS = read_terms_from_file(STRONG_EXCLUDE_TERMS_PATH)

In [3]:
print(KEY_TERMS)
print(EXCLUDE_TERMS)
print(STRONG_EXCLUDE_TERMS)

['gravida', 'pre natal', 'pre natal', 'prenatal', 'gravidez', 'parto', 'contracao', 'contracoes', 'embriao', 'feto', 'gestacao', 'colostro', 'amamentacao', 'leite materno', 'amamentar', 'teste do pezinho', 'teste do olhinho', 'teste da orelhinha', 'loquio', 'mamas endurecidas', 'seios endurecidos', 'seios empedrados', 'cesarea', 'cesaria', 'cesariana', 'coracao do bebe', 'sexo do bebe', 'prematuro', 'batimento fetal', 'gestante', 'recem nascido', 'recem nascida', 'bebe mexendo', 'movimento fetal', 'puerperio', 'gestacional', 'ingurgitamento mamario', 'semanas de gravidez', 'meses de gestacao', 'atraso menstrual', 'atraso da menstruacao', 'teste de gravidez', 'maternidade', 'nidacao', 'pos parto', 'pos parto']
['bebe reborn', 'bebes reborn', 'bebes reborns', 'boneca reborn', 'bonecas reborn', 'maternidade de bonecas', 'novelinha', 'globoplay novelas', 'episodio dublado', 'filme romantico', 'romance e drama', 'serie completa', 'essa vida e minha', 'garota do momento', 'itwelcometoderry',

In [4]:
# uses lingua for language detection, known for working well even with small comments
def is_portuguese(comment: str) -> bool:
    lang = detector.detect_language_of(str(comment).strip())

    return lang == Language.PORTUGUESE

## Channels

In [5]:
df_channels = pd.read_csv(CHANNELS_PATH, usecols=COLS_CHANNELS)
df_channels.head()

,channel_id,title,description,published_at,country,view_count,comment_count,subscriber_count,video_count,keywords,profile_picture_url
0,UCIVD2vf5odi1wQ-_Lp7TNoQ,Mariblush,Seja bem vindaaa (ooo) !!! \n✨Se preparem que ...,2020-11-17T04:27:45.32055Z,BR,245522,0,1290,145,NaN,https://yt3.ggpht.com/lwN5I1Z8jsDBe9LhiIXgfsIR...
1,UCkOy-wA0D6iCnC04-RRylwA,Canal da Fer - Fernanda Teles,"SOBRE MIM 💁\n\nOiiiii, eu sou a Fernanda mas p...",2015-03-28T11:58:40Z,BR,5270710,0,43800,352,"""rotina de dona de casa"" ""vida de casada"" ""vid...",https://yt3.ggpht.com/g4sm70eY68JPVPGfCnqjmoEs...
2,UCyGYiuG-4dVjJxpVkE9etuw,Luana Felício,Oh Eu Aquii !! \n\n Meu Nome é Luana Felício...,2009-05-24T23:50:37Z,BR,2968963,0,29600,198,Lifestyle Vlog Beleza Gestação Maternidade,https://yt3.ggpht.com/UdAwcYhzidun2e7V6o-BAoSR...
3,UC4KLdtHRXb-B78LTb_50LQQ,Vitória Magalhães,\nInstagram:@mavi_magha\nFacebook: Vitória Ma...,2018-10-18T11:05:26Z,NaN,1711,0,69,8,NaN,https://yt3.ggpht.com/ytc/AIdro_nmUOT0eL93_KFb...
4,UCLHr3ZsH_wcVuB-ZqciAJsQ,Escola de Ultrassom | Ultrassonografia,Faça a sua matrícula 👇🏻,2024-10-28T18:10:34.058141Z,BR,177484,0,3170,66,"""Escola de Ultrassom"" Ultrassom Ultrassonograf...",https://yt3.ggpht.com/8-q-GJ1_77cTz7fjdBy1Te9Q...


In [6]:
# saving number of channels for later comparison
initial_channels_number = df_channels['channel_id'].nunique()

In [7]:
# passing to datetime and ordering
df_channels['published_at'] = pd.to_datetime(df_channels['published_at'], utc=True, format='mixed')
df_channels['published_at'] = df_channels['published_at'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
df_channels.sort_values(by='published_at', inplace=True)

In [8]:
df_channels.head()

,channel_id,title,description,published_at,country,view_count,comment_count,subscriber_count,video_count,keywords,profile_picture_url
4999,UC6wBro4B4pf9xnBh9Xi2zcQ,David Hoffman,I post videos at 1pm Pacific during weekdays a...,2005-09-27 14:46:14,US,484979130,0,1350000,3609,"""David Hoffman"" ""bluegrass music"" ""Vietnam War...",https://yt3.ggpht.com/ytc/AIdro_nb3L05EbCm-QKo...
3383,UC_vcSdkPQSms9LFi4DBvJew,Dr. Abdom Fora,"Hola, soy el Dr. Fora, te doy la bienvenida a ...",2005-12-23 01:24:52,PE,1618011075,0,3390000,144,"""dentist procedure"" ""dentist teeth cleaning pr...",https://yt3.ggpht.com/Hi16tet9KY-br3p8b4aVNXAi...
7121,UCOa-WaNwQaoyFHLCDk7qKIw,Flamengo TV,O canal oficial do Clube de Regatas do Flamengo.,2006-02-27 01:11:12,BR,1237320262,0,8070000,13969,"flatv ""fla tv"" flamengo Mengo Mengão Futebol ""...",https://yt3.ggpht.com/p-lbVutCzOzpPvdVF3S6vymn...
13433,UCz6Pr0KLinJBZxhDHmBAu6g,Capricho,Manifeste. Desobedeça. Seja você.\n,2006-03-12 12:15:34,BR,245644668,0,1560000,3564,NaN,https://yt3.ggpht.com/aeuldEGvgzngLeTa7IdS50oE...
8704,UCgKq-YU2eFrbByW8I6OiiYg,Cinthia Reyes,Respondo preguntas cotidianas leyendo evidenci...,2006-03-21 20:34:38,MX,41211212,0,281000,1876,"""divulgación de ciencia"" física química cienci...",https://yt3.ggpht.com/61E1mngfDmf2ruWe41KKzXHg...


In [9]:
df_channels.isna().sum()

channel_id                0
title                     0
description            1135
published_at              0
country                3839
view_count                0
comment_count             0
subscriber_count          0
video_count               0
keywords               6003
profile_picture_url       0
dtype: int64

In [10]:
# filter only channels relevant to brazilian context
df_channels = df_channels[(df_channels['country'] == 'BR') | (df_channels['country'].isnull())]
df_channels.isna().sum()

channel_id                0
title                     0
description            1038
published_at              0
country                3839
view_count                0
comment_count             0
subscriber_count          0
video_count               0
keywords               5491
profile_picture_url       0
dtype: int64

In [11]:
# keep only unique info about channels
df_channels.drop_duplicates(subset=['channel_id'], inplace=True)
df_channels.duplicated().sum()

0

In [12]:
# fill missing values for description and keywords using a empty string
# and droping any missing data that remains
df_channels.fillna(
    {
        'description': '',
        'keywords': '',
        'country': 'Unknown'
    },
    inplace=True
)
df_channels.dropna(
    subset=['channel_id', 'title', 'published_at'],
    inplace=True
)

# ids for further filtering
channels_ids = df_channels['channel_id'].tolist()

In [13]:
df_channels.drop(columns=['country'], inplace=True)

In [14]:
print(f"After the initial treatment, {df_channels['channel_id'].nunique()} channels remained, out of {initial_channels_number}")

After the initial treatment, 7571 channels remained, out of 9134


In [15]:
print(f'DataFrame containing channels information was saved at {CHANNELS_CLEANED_PATH}')
df_channels.to_csv(CHANNELS_CLEANED_PATH, index=False, encoding='utf-8')

DataFrame containing channels information was saved at ../data/cleaned/cleaned_channels_info.csv


## Videos

In [16]:
df_videos = pd.read_csv(VIDEOS_PATH, usecols=COLS_VIDEOS)
df_videos.head()

,video_id,title,description,channel_id,published_at,category_id,tags,view_count,like_count,comment_count,duration,licensed_content,is_made_for_kids,thumbnail_url,default_audio_language,default_language,topicCategories,is_short
0,2MT-6VW_UVw,Descobrindo que estou grávida - VLOG,Depois de muitoa anos ate desaprendi a ver um ...,UCIVD2vf5odi1wQ-_Lp7TNoQ,2025-12-25T14:44:17Z,22,[],242,14,5,PT10M37S,False,False,https://i.ytimg.com/vi/2MT-6VW_UVw/hqdefault.jpg,NaN,pt-PT,['https://en.wikipedia.org/wiki/Lifestyle_(soc...,False
1,UelsGRUlIeI,DESCOBRINDO A GRAVIDEZ ANTES DO ATRASO,Como eu descobri minha gravidez 5 dias antes d...,UCkOy-wA0D6iCnC04-RRylwA,2025-12-12T21:04:51Z,24,"['Gravidez', 'Descobrindo a gravidez', 'estou ...",171,16,3,PT4M1S,True,False,https://i.ytimg.com/vi/UelsGRUlIeI/hqdefault.jpg,pt,pt-PT,"['https://en.wikipedia.org/wiki/Health', 'http...",False
2,NPjMZxE2Wn8,Tenho certeza que to gravida - Vamos fazer um ...,Vem descobrir comigo se eu estou gravida de novo,UCyGYiuG-4dVjJxpVkE9etuw,2025-12-26T20:26:02Z,24,[],66,5,1,PT9M12S,False,False,https://i.ytimg.com/vi/NPjMZxE2Wn8/hqdefault.jpg,NaN,pt-PT,"['https://en.wikipedia.org/wiki/Health', 'http...",False
3,G3Ut7J9cW5w,Reações contando que estou grávida! Já te amam...,"Nosso Sonho, Nosso Milagre, Nosso Bebê\n\nHá t...",UC4KLdtHRXb-B78LTb_50LQQ,2025-12-11T15:52:14Z,22,[],159,7,4,PT7M50S,False,False,https://i.ytimg.com/vi/G3Ut7J9cW5w/hqdefault.jpg,NaN,pt-PT,['https://en.wikipedia.org/wiki/Lifestyle_(soc...,False
4,kLlNLJJKMgU,Você está sendo enganado por esse Diagnóstico?...,Entenda como reconhecer uma gravidez Anembrion...,UCLHr3ZsH_wcVuB-ZqciAJsQ,2025-12-08T11:01:34Z,28,"['Escola de Ultrassom', 'Ultrassom', 'gestação...",3449,70,3,PT59S,False,False,https://i.ytimg.com/vi/kLlNLJJKMgU/hqdefault.jpg,pt-BR,pt-BR,"['https://en.wikipedia.org/wiki/Health', 'http...",True


In [17]:
initial_videos_number = df_videos['video_id'].nunique()
df_videos.shape

(17512, 18)

In [18]:
# passing to datetime and ordering
df_videos['published_at'] = pd.to_datetime(df_videos['published_at'], utc=True, format='mixed')
df_videos['published_at'] = df_videos['published_at'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
df_videos.sort_values(by='published_at', inplace=True)

In [19]:
df_videos.drop_duplicates(subset=['video_id'], inplace=True)
df_videos.duplicated().sum()

0

In [20]:
df_videos.isna().sum()

video_id                     0
title                        0
description               5457
channel_id                   0
published_at                 0
category_id                  0
tags                         0
view_count                   0
like_count                   0
comment_count                0
duration                     0
licensed_content             0
is_made_for_kids             0
thumbnail_url                0
default_audio_language    7650
default_language             0
topicCategories              0
is_short                     0
dtype: int64

In [21]:
# bool cols for filtering portuguese content
df_videos['audio_is_pt'] = df_videos['default_audio_language'].str.startswith('pt', na=False)
df_videos['language_is_pt'] = df_videos['default_language'].str.startswith('pt', na=False)

# filtering based on channel region
df_videos = df_videos[df_videos['channel_id'].isin(channels_ids)]

# filtering based on video language
df_videos = df_videos[(df_videos['audio_is_pt']) | (df_videos['language_is_pt'])]

In [22]:
# fill missing description with a empty string and drop missing values
df_videos.fillna({'description': ''}, inplace=True)
df_videos = df_videos.dropna(subset=['video_id', 'title', 'channel_id', 'published_at'])

In [23]:
df_videos.shape

(14309, 20)

In [24]:
# filters videos where the title/description mentions, at least, one of the predefined key terms
key_pattern = r'\b(?:' + '|'.join(re.escape(term) for term in KEY_TERMS) + r')\b'
title_exclude_pattern = r'\b(?:' + '|'.join(re.escape(term) for term in EXCLUDE_TERMS) + r')\b'
strong_exclude_pattern = r'\b(?:' + '|'.join(re.escape(term) for term in STRONG_EXCLUDE_TERMS) + r')\b'

df_videos['title_clean'] = df_videos['title'].apply(lambda x: padronize_text(x.lower()))
df_videos['title_description'] = df_videos['title'] + ' ' + df_videos['description']
df_videos['title_description'] = df_videos['title_description'].apply(lambda x: padronize_text(x.lower()))

df_videos = df_videos[
    df_videos['title_description'].str.contains(key_pattern, case=False, na=False) &
    ~df_videos['title_clean'].str.contains(title_exclude_pattern, case=False, na=False) &
    ~df_videos['title_description'].str.contains(strong_exclude_pattern, case=False, na=False)
]

# ids for further filtering
videos_ids = df_videos['video_id'].tolist()

In [25]:
df_videos.drop(columns=['audio_is_pt', 'language_is_pt', 'default_audio_language', 'default_language', 'title_clean', 'title_description'], inplace=True)

In [26]:
print(f"After the initial treatment, {df_videos['video_id'].nunique()} videos remained out of {initial_videos_number}")

After the initial treatment, 3602 videos remained out of 17511


In [27]:
print(f'DataFrame containing videos information was saved at {VIDEOS_CLEANED_PATH}')
df_videos.to_csv(VIDEOS_CLEANED_PATH, index=False, encoding='utf-8')

DataFrame containing videos information was saved at ../data/cleaned/cleaned_videos_info.csv


## Comments

In [28]:
# using polasr to read comments of videos filtered in aforementioned stage
df_comments = (
    pl.scan_csv(COMMENTS_PATH, try_parse_dates=True)
    .select(COLS_COMMENTS)
    .filter(pl.col('video_id').is_in(videos_ids))
    .collect()
)

df_comments.head()

video_id,comment_id,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,like_count,is_reply,parent_id,channel_id
str,str,str,str,str,str,"datetime[μs, UTC]",i64,bool,str,str
"""2MT-6VW_UVw""","""UgzDf_p6KOXykHcsoWB4AaABAg""","""https://yt3.ggpht.com/ytc/AIdr…","""http://www.youtube.com/@camila…","""UC1tfFQ6FrF7TcKmZOUKK2Tg""","""Presente de Deus, parabéns ❤""",2025-12-26 00:55:59 UTC,0,false,null,"""UCIVD2vf5odi1wQ-_Lp7TNoQ"""
"""2MT-6VW_UVw""","""UgwX-T4jOv0032Xj_XJ4AaABAg""","""https://yt3.ggpht.com/21hwjZqm…","""http://www.youtube.com/@gabrie…","""UCdNyGGx8dJQ8-kbmUqLu8wA""","""Parabénssss❤""",2025-12-25 15:53:07 UTC,0,false,null,"""UCIVD2vf5odi1wQ-_Lp7TNoQ"""
"""2MT-6VW_UVw""","""Ugw13H9Fj1mWqoL_W7Z4AaABAg""","""https://yt3.ggpht.com/ytc/AIdr…","""http://www.youtube.com/@Mateus…","""UC5LXuDtzNwGJplUsk4FFcLw""","""Tem certeza ? Não quer fazer o…",2025-12-25 15:01:11 UTC,0,false,null,"""UCIVD2vf5odi1wQ-_Lp7TNoQ"""
"""2MT-6VW_UVw""","""UgyFV4Oq8581G8g9MsB4AaABAg""","""https://yt3.ggpht.com/7w0dfyF1…","""http://www.youtube.com/@partiu…","""UCKvqe76CvzJc9eVqcmk2CJg""","""Parabéns Mari, Que Venha com m…",2025-12-25 14:56:38 UTC,0,false,null,"""UCIVD2vf5odi1wQ-_Lp7TNoQ"""
"""2MT-6VW_UVw""","""UgyFV4Oq8581G8g9MsB4AaABAg.AR9…","""https://yt3.ggpht.com/lwN5I1Z8…","""http://www.youtube.com/@maribl…","""UCIVD2vf5odi1wQ-_Lp7TNoQ""","""Amém!! Obrigada querida 🥰❤️""",2025-12-25 15:15:32 UTC,0,true,"""UgyFV4Oq8581G8g9MsB4AaABAg""","""UCIVD2vf5odi1wQ-_Lp7TNoQ"""


In [29]:
initial_comments_number = (
    pl.scan_csv(COMMENTS_PATH)
    .select(pl.col('comment_id').n_unique())
    .collect()
    .item()
)

In [30]:
df_comments.shape

(362274, 11)

In [31]:
# converting timezone
df_comments = df_comments.with_columns(
    pl.col('published_at')
    .dt.convert_time_zone('America/Sao_Paulo')
    .dt.replace_time_zone(None)
)

In [32]:
df_comments = (
    df_comments
    .unique(subset=["comment_id"])
    .with_columns(
        pl.col("comment")
        .cast(pl.String)
        .str.replace_all(r"https?://\S+|www\.\S+", "")
        .str.replace_all(r"@\w+", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )
    .filter(
        (pl.col("comment").str.count_matches(r"[A-Za-zÀ-ÿ]") >= 3) &
        (pl.col("comment").str.split(" ").list.len() >= 3)
    )
    .filter(
        pl.col("comment").map_elements(
            is_portuguese,
            return_dtype=pl.Boolean
        )
    )
)

In [33]:
print(f"After the initial treatment, {df_comments['comment_id'].n_unique()} comments remained out of {initial_comments_number}")

After the initial treatment, 225616 comments remained out of 6742440


In [34]:
print('Final values for database:')
print(f'Number of channels: {df_comments["channel_id"].n_unique()}')
print(f'Number of videos: {df_comments["video_id"].n_unique()}')
print(f'Number of comments: {df_comments["comment_id"].n_unique()}')
print(f'Number of authors: {df_comments["author_channel_id"].n_unique()}')

Final values for database:
Number of channels: 1495
Number of videos: 3330
Number of comments: 225616
Number of authors: 172426


In [35]:
print(f'DataFrame containing comments information was saved at {COMMENTS_CLEANED_PATH}')
df_comments.write_csv(COMMENTS_CLEANED_PATH)

DataFrame containing comments information was saved at ../data/cleaned/cleaned_comments_info.csv
